# AdMarket Arena — Advertiser GRPO Training

Trains the **trained LLM advertiser** (Theme 1 + Theme 2 headline) on the AdMarket Arena env via TRL GRPO + Unsloth 4-bit LoRA.

**T4 (Colab) tuned config** — base model: Qwen2.5-3B-Instruct (4-bit NF4 + LoRA r=16), GRPO `num_generations=4`, 64-token output cap, ~4 hr expected for 50-80 GRPO steps. Curriculum scheduler escalates `arena_easy → arena_medium → arena_hard` when mean episode reward > 0.30 for 10 consecutive rollouts (master Section 3.1.1).

Reuses Plan 3's shared `training_callbacks.py` (CSV + JSONL + best-checkpoint tracker). Plan 2's `train_oversight.ipynb` reuses the same callback with a different run name.

**Outputs**:
- `checkpoints/advertiser_run/checkpoint-<N>/` LoRA adapters (every 10 GRPO steps)
- `checkpoints/advertiser_run/best/` copy of the best by mean weekly_roas over last 5 validation episodes
- `logs/training_run_advertiser_grpo_*.csv` tamper-evident receipt
- `logs/episodes/advertiser_grpo/ep_step*.jsonl` validation episode dumps every 10 steps
- HF Hub push to `<org>/admarket-advertiser-qwen2.5-3b-grpo`

**Reward = episode return**: per-step engagement + daily ROAS bonus + weekly ROAS bonus (master Section 4). Per master Section 6, one prompt = one rollout step (the agent emits a single `AuctionAction` JSON per impression opportunity); the env supplies the multi-step composition under the hood.

## 1. Install dependencies

In [ ]:
%%capture
# Colab T4 install (Apr 2026).
#
# Hard-won lessons:
#   * Do NOT pin transformers / accelerate / datasets — Unsloth's latest needs
#     transformers>=4.47 (for `CompileConfig`); older pins silently downgrade
#     the wheel pip just installed and Unsloth then ImportErrors at import time.
#   * bitsandbytes must be >=0.45.0 — 0.44 imports the removed `triton.ops`
#     and ships no CUDA 12.8 wheel for current Colab runtimes.
#   * Install Unsloth LAST so its dependency resolver gets the final say
#     over transformers/accelerate/peft versions.
!pip install -U "trl>=0.12" "peft>=0.13"
!pip install -U "bitsandbytes>=0.45.0"
!pip install -U pydantic matplotlib
# Required by models.py / server.arena_env (provides openenv.core.env_server.types)
!pip install -U "openenv-core>=0.2.1"
# Unsloth pulls compatible transformers / accelerate / xformers wheels itself.
!pip install -U unsloth

## 2. Mount drive / clone repo

If you've copied the repo into the Colab runtime already, skip the clone and just `cd` into it. Otherwise pull from the team repo so the latest `competitors.py`, `training_callbacks.py`, `curriculum_scheduler.py`, `scripts/advertiser_eval.py` are available.

In [ ]:
import os, sys
REPO_PATH = '/content/AdMarket-Arena'  # cloned from ritz-gupta/AdMarket-Arena
if not os.path.exists(REPO_PATH):
    !git clone -b play2 https://github.com/ritz-gupta/AdMarket-Arena.git $REPO_PATH
%cd $REPO_PATH
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

## 3. Imports + config

In [ ]:
import os
import random
import statistics
import textwrap
from pathlib import Path
from typing import Dict, List

import torch
from datasets import Dataset

from competitors import (
    ADVERTISER_SYSTEM_PROMPT,
    _format_observation_for_advertiser,
    parse_llm_advertiser_action,
)
from curriculum_scheduler import make_advertiser_curriculum
from models import AuctionAction, AuctionObservation
from training_callbacks import make_arena_callback

# --- knobs ---
BASE_MODEL          = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LEN         = 4096
LORA_RANK           = 16
OUTPUT_DIR          = Path('checkpoints/advertiser_run')
MAX_NEW_TOKENS      = 64
NUM_GENERATIONS     = 4
LEARNING_RATE       = 1e-5
MAX_STEPS           = 80           # ~4 hr on T4; set 20 for a quick smoke-test
SAVE_EVERY_STEPS    = 10
LOG_EVERY_STEPS     = 1
RUN_NAME            = 'advertiser_grpo_qwen3b'
HF_HUB_REPO_ID      = 'ritz-gupta/admarket-advertiser-qwen2.5-3b-grpo'
SEED                = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
Path('logs').mkdir(parents=True, exist_ok=True)
Path('logs/episodes').mkdir(parents=True, exist_ok=True)

random.seed(SEED); torch.manual_seed(SEED)
print('Config OK.  current curriculum tier = arena_easy')

## 4. Load model with Unsloth (4-bit LoRA)

Same recipe as `train_oversight.ipynb` — shared base weights are cached on disk so the second notebook (oversight) loads in seconds when run sequentially after this one.

In [ ]:
import unsloth  # safe to re-import; cheap if already loaded

model, tokenizer = unsloth.FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)
model = unsloth.FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
print(f'Model loaded: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params (LoRA).')

## 5. Curriculum scheduler

The scheduler drives both the env tier (`arena_easy → arena_medium → arena_hard`) and the rollout config (impression count, opponent slate). Promotion fires when mean rollout reward > 0.30 for 10 consecutive rollouts. The TrainerCallback bridge (`scheduler.as_callback()`) feeds rollout rewards in automatically on every TRL `on_log` event so the notebook only has to call `add_callback` once.

In [ ]:
_current_tier = {'name': 'arena_easy'}

def _on_promote(old: str, new: str, step: int) -> None:
    print(f'[curriculum] step={step} promoting {old} -> {new}')
    _current_tier['name'] = new

scheduler = make_advertiser_curriculum(on_promote=_on_promote, promotion_threshold=0.30, required_streak=10)
print('Curriculum scheduler ready. Initial tier:', scheduler.current_tier)

## 6. Build dataset (one prompt = one auction step)

GRPO trains on (prompt, group of completions, reward) triples. Each prompt is a rendered `AuctionObservation` for the trained advertiser at one auction step; the reward function below is the **per-step engagement + daily/weekly ROAS** composite from master Section 4.

We roll out episodes through the real `server.arena_env.AdMarketArenaEnvironment` with `ArenaPacingAgent` driving the trained-advertiser slot — the env steps deterministically forward and we capture each `AuctionObservation` it emits as a `(prompt, observation_json)` row. GRPO then samples `num_generations` completions per prompt and scores each with the synthetic per-step reward in §7.

In [ ]:
from baseline import ArenaPacingAgent
from server.arena_env import AdMarketArenaEnvironment

BUILD_EPISODES_PER_TIER = 20      # ~50-350 prompts/episode depending on tier
MAX_EXAMPLES = 1000

def _render_prompt(observation: AuctionObservation) -> str:
    return tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': ADVERTISER_SYSTEM_PROMPT},
            {'role': 'user', 'content': _format_observation_for_advertiser(observation)},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )

def _episode_to_examples(env: AdMarketArenaEnvironment, episode_seed: int, tier: str) -> List[Dict[str, str]]:
    """Roll out one real arena episode; capture each (prompt, observation) seen
    by the trained-advertiser slot. ArenaPacingAgent drives the env forward so
    the loop terminates; the captured prompts are what TRL trains on."""
    pacing = ArenaPacingAgent()
    captured: List[Dict[str, str]] = []
    obs = env.reset(task=tier, seed=episode_seed)
    while not obs.done:
        captured.append({
            'prompt': _render_prompt(obs),
            'observation_json': obs.model_dump_json(),
            'tier': tier,
        })
        obs = env.step(pacing.act(obs))
    return captured

env = AdMarketArenaEnvironment()
examples: List[Dict[str, str]] = []
for ep in range(BUILD_EPISODES_PER_TIER):
    examples.extend(_episode_to_examples(env, SEED + ep, scheduler.current_tier))
    if len(examples) >= MAX_EXAMPLES:
        break

random.Random(SEED).shuffle(examples)
split_at = max(1, int(len(examples) * 0.9))
train_ds = Dataset.from_list(examples[:split_at])
val_ds   = Dataset.from_list(examples[split_at:])
print(f'Built {len(examples)} step-examples ({scheduler.current_tier}).  train={len(train_ds)}  val={len(val_ds)}')
print(train_ds[0]['prompt'][:400], '...')

## 7. Reward function — per-step + daily/weekly ROAS proxy

Master Section 4 defines the full reward as `per_step_reward + daily_reward (boundaries) + weekly_reward (episode end)`. For per-step GRPO training we collapse this into a single scalar per (prompt, completion) by:

  - parsing the LLM's JSON action,
  - estimating win probability against the recorded observation's competitive context (recent clearing prices),
  - computing per-step engagement (clicks - clearing-price waste - skip nudge),
  - adding a small pacing bonus when daily spend is on track.

This proxy is intentionally light so GRPO can score every (prompt, completion) cheaply. The full multi-tier reward (per-step + daily + weekly) is what `arena_env` fires during the real validation rollout in §8 — that's what the curriculum scheduler and best-checkpoint tracker consume.

In [ ]:
def _per_step_reward(observation: AuctionObservation, action: AuctionAction) -> float:
    """Per-step proxy reward used during GRPO training.

    Light enough to score every (prompt, completion) cheaply. The full
    multi-tier reward (per-step engagement + daily pacing + weekly ROAS)
    fires inside ``arena_env`` during the validation rollout — that is
    what the eval pipeline + best-checkpoint tracker consume.
    """
    if action.skip:
        return 0.02  # tiny budget-preservation nudge
    floor = max(observation.floor_price, 1e-3)
    if action.bid_amount < floor:
        return -0.05
    if action.bid_amount > observation.daily_budget_remaining + 0.1:
        return -0.5  # invalid bid (over budget)
    market = (sum(observation.recent_clearing_prices) / len(observation.recent_clearing_prices)
              if observation.recent_clearing_prices else floor + 0.2)
    win_prob = max(0.0, min(1.0, (action.bid_amount - market) / max(market, 1e-3) + 0.5))
    expected_clearing = max(floor, market)
    expected_revenue = 1.5 * win_prob * 0.18  # base CTR ~ 0.18 in the synthetic engine
    expected_clearing_paid = win_prob * expected_clearing
    margin = expected_revenue - expected_clearing_paid
    daily_budget = max(1e-3, observation.spent_so_far_today + observation.daily_budget_remaining)
    pacing_bonus = 0.05 if observation.spent_so_far_today < 0.5 * daily_budget else 0.0
    return float(margin + pacing_bonus)

def advertiser_reward_fn(prompts, completions, observation_json, **_):
    rewards: List[float] = []
    for completion, obs_json in zip(completions, observation_json):
        text = completion if isinstance(completion, str) else completion[-1].get('content', '')
        observation = AuctionObservation.model_validate_json(obs_json)
        action = parse_llm_advertiser_action(text, n_creatives=len(observation.available_creatives))
        rewards.append(_per_step_reward(observation, action))
    return rewards

# Sanity-check on the validation set.
_demo_obs = AuctionObservation.model_validate_json(val_ds[0]['observation_json'])
_demo_action = parse_llm_advertiser_action('{"skip": false, "bid_amount": 0.8, "creative_id": 0}', n_creatives=len(_demo_obs.available_creatives))
print('demo per-step reward:', _per_step_reward(_demo_obs, _demo_action))

## 8. Smoke-test logging on first 2 GRPO steps

**Critical insurance step.** Verify CSV + per-step custom metrics fire correctly before launching the long run. A broken logger costs 4 hours of GPU time.

In [ ]:
_validation_env = AdMarketArenaEnvironment()
_validation_pacing = ArenaPacingAgent()

def _validation_metrics(training_step: int) -> dict:
    """Env health-check using a deterministic pacing baseline.

    The headline reward curve is plotted from TRL's ``reward`` column
    (mean GRPO reward across the batch — i.e. the *trained* policy's
    actual reward signal), so this function is **not** the source of
    that plot. Every 10 steps it rolls the pacing baseline through the
    current curriculum tier and emits ``pacing_baseline_*`` columns:
    useful as a stable reference line for sanity-checking the env and
    as a comparison series in plots. The post-training eval cell (§11)
    runs the trained checkpoint through the full 3-mode harness."""
    if training_step % 10 != 0:
        return {}
    obs = _validation_env.reset(task=scheduler.current_tier, seed=10_000 + training_step)
    episode_return = 0.0
    while not obs.done:
        obs = _validation_env.step(_validation_pacing.act(obs))
        episode_return += float(obs.reward or 0.0)
    return {
        'pacing_baseline_weekly_roas': float(_validation_env.state.weekly_roas),
        'pacing_baseline_episode_return': round(episode_return, 4),
    }

callback = make_arena_callback(
    run_name=RUN_NAME,
    log_dir='logs',
    checkpoint_root='checkpoints',
    episode_dump_every=10,
    save_checkpoint_every=SAVE_EVERY_STEPS,
    best_metric_name='reward',
    higher_is_better=True,
    custom_metrics_fn=_validation_metrics,
)
print('Callback ready. Logs → logs/training_run_advertiser_grpo_qwen3b_*.csv')

## 9. GRPO trainer + run

Two callbacks: the unified observability callback (W&B + CSV + JSONL + best-tracker) and the curriculum scheduler bridge that promotes the env tier on the fly.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir=str(OUTPUT_DIR),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_generations=NUM_GENERATIONS,
    max_prompt_length=MAX_SEQ_LEN - MAX_NEW_TOKENS,
    max_completion_length=MAX_NEW_TOKENS,
    max_steps=MAX_STEPS,
    save_steps=SAVE_EVERY_STEPS,
    logging_steps=LOG_EVERY_STEPS,
    bf16=False,
    fp16=True,
    optim='adamw_8bit',
    report_to=[],          # no W&B — CSV logs only
    remove_unused_columns=False,
    seed=SEED,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[advertiser_reward_fn],
    args=config,
    train_dataset=train_ds,
)
trainer.add_callback(callback)
trainer.add_callback(scheduler.as_callback())
trainer.train()

## 10. Validation pass + best-checkpoint selection

Best checkpoint = max mean `weekly_roas` over the last 5 validation episodes. The callback already snapshots `checkpoints/advertiser_run/best/` whenever a higher value is seen during training; this cell is the explicit final dump that the eval + HF Hub push read from.

In [ ]:
unsloth.FastLanguageModel.for_inference(model)

def _generate(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

per_step_reward_samples: List[float] = []
for example in val_ds.select(range(min(5, len(val_ds)))):
    text = _generate(example['prompt'])
    obs = AuctionObservation.model_validate_json(example['observation_json'])
    action = parse_llm_advertiser_action(text, n_creatives=len(obs.available_creatives))
    per_step_reward_samples.append(_per_step_reward(obs, action))
mean_recent = statistics.mean(per_step_reward_samples) if per_step_reward_samples else 0.0
print(f'Final-pass mean per-step reward over {len(per_step_reward_samples)} samples: {mean_recent:.3f}')

FINAL_PATH = OUTPUT_DIR / 'final'
model.save_pretrained(str(FINAL_PATH))
tokenizer.save_pretrained(str(FINAL_PATH))
print(f'Saved final checkpoint to {FINAL_PATH}')
print(f'Best (auto-tracked) at:   {OUTPUT_DIR / "best"}')

## 11. Run the 3-mode advertiser eval (standard / edge / selfplay)

Mirrors `python -m scripts.advertiser_eval`. Uses the **best** checkpoint as the trained policy and **final** as the v1 selfplay opponent (or pass an earlier `checkpoint-10/` snapshot for stronger v2-vs-v1 contrast).

In [ ]:
from scripts.advertiser_eval import run_advertiser_eval

EVAL_RESULTS_PATH = Path('results/advertiser_eval.json')
EVAL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = OUTPUT_DIR / 'best'
EARLY_CHECKPOINT = OUTPUT_DIR / 'checkpoint-10'   # selfplay v1 opponent (None falls back to mock)

payload = run_advertiser_eval(
    task_name='arena_hard',
    checkpoint=str(BEST_CHECKPOINT),
    opponent_checkpoint=str(EARLY_CHECKPOINT) if EARLY_CHECKPOINT.exists() else None,
    n_standard=20,
    n_edge_per_sub=10,
    n_selfplay=10,
    out_path=EVAL_RESULTS_PATH,
    include_baselines=True,
)
for mode, m in payload['per_mode'].items():
    print(f"[eval] {mode:>9s}  weekly_roas={m['weekly_roas_mean']:.3f}  bid_precision={m['bid_precision_mean']:.3f}  depl_day={m['budget_depletion_day_mean']:.2f}  fatigue_sens={m['fatigue_sensitivity_mean']:.3f}")

## 12. Generate the 5 advertiser PNGs

Pure CSV → PNG; matches the `make_plots.py` invocation in the eval script. PNGs land in `assets/plots/` and the source CSVs in `assets/data/`.

In [ ]:
from scripts.make_plots import (
    plot_bid_precision_hist,
    plot_budget_depletion_comparison,
    plot_fatigue_sensitivity,
    plot_loss_curve,
    plot_reward_curve,
)

PLOTS_DIR = Path('assets/plots'); PLOTS_DIR.mkdir(parents=True, exist_ok=True)
advertiser_csv = next(Path('logs').glob(f'training_run_{RUN_NAME}_*.csv'), None)
if advertiser_csv is None:
    raise FileNotFoundError(
        f'No training CSV at logs/training_run_{RUN_NAME}_*.csv — did the GRPO cell above run to completion?'
    )

# reward_key='reward' targets TRL's per-step GRPO mean reward column
# (the trained policy's actual signal). The legacy default
# 'episode_return_total' was sourced from the deterministic pacing
# baseline rollout in §8 and so could not improve with training.
for fn, kwargs in [
    (plot_reward_curve,        dict(advertiser_csv=advertiser_csv, baseline_csvs={}, out_path=PLOTS_DIR / 'reward_curve.png', reward_key='reward')),
    (plot_loss_curve,          dict(advertiser_csv=advertiser_csv, out_path=PLOTS_DIR / 'loss_curve.png')),
    (plot_bid_precision_hist,  dict(eval_json=EVAL_RESULTS_PATH, advertiser_csv=advertiser_csv, out_path=PLOTS_DIR / 'bid_precision_hist.png')),
    (plot_budget_depletion_comparison, dict(eval_json=EVAL_RESULTS_PATH, out_path=PLOTS_DIR / 'budget_depletion_comparison.png')),
    (plot_fatigue_sensitivity, dict(advertiser_csv=advertiser_csv, out_path=PLOTS_DIR / 'fatigue_sensitivity_corr.png')),
]:
    out = fn(**kwargs)
    print(f'  -> {out}')

## 13. Push to HuggingFace Hub

Auto-populated model card includes: training config, env config, reward decomposition formula, eval table from `advertiser_eval.json`, public W&B URL, training cost (T4-hours).

In [ ]:
# from huggingface_hub import HfApi, login
# login()  # paste HF token when prompted

# BEST_PATH = OUTPUT_DIR / 'best'
# model.push_to_hub(HF_HUB_REPO_ID, private=False)
# tokenizer.push_to_hub(HF_HUB_REPO_ID, private=False)

# _eval_summary = '\n'.join(
#     f"- {mode}: weekly_roas={m['weekly_roas_mean']:.3f}, bid_precision={m['bid_precision_mean']:.3f}, depl_day={m['budget_depletion_day_mean']:.2f}, fatigue_sens={m['fatigue_sensitivity_mean']:.3f}"
#     for mode, m in payload.get('per_mode', {}).items()
# )

# model_card = textwrap.dedent(f"""\
# ---
# language: en
# license: apache-2.0
# library_name: peft
# tags:
# - grpo
# - unsloth
# - multi-agent
# - ad-tech
# - long-horizon
# base_model: {BASE_MODEL}
# ---

# # AdMarket Arena — Trained Advertiser

# GRPO-trained LoRA adapter for the AdMarket Arena env's *advertiser* role. Bids, paces budget, and rotates creatives across a 7-day campaign in a 5-advertiser second-price auction.

# **Training**: {MAX_STEPS} GRPO steps, num_generations={NUM_GENERATIONS}, lr={LEARNING_RATE}, on Colab T4.

# **Eval (robustness table)**:
# {_eval_summary}

# **Env repo**: https://github.com/ritz-gupta/AdMarket-Arena
# """)
# Path('MODEL_CARD.md').write_text(model_card)
# HfApi().upload_file(path_or_fileobj='MODEL_CARD.md', path_in_repo='README.md', repo_id=HF_HUB_REPO_ID, repo_type='model')
# print('Pushed to', HF_HUB_REPO_ID)